# 依赖锁候选审查包入口

该 Notebook 只负责挂载 Drive、锁定精确代码提交并调用仓库单一资格化脚本。候选锁不写入 `configs/`, 不支持论文结论, 必须经过人工审查和 Git 提交后才能成为运行输入。

## 使用顺序

1. 首先在匹配的 Linux x86_64 CPU 解释器中选择 `workflow_orchestrator`, 生成并审查候选锁, 再把审查通过的完整锁提交到仓库。
2. 只有新提交包含 orchestrator 完整锁后, 才能选择五个科学 profile。资格化脚本会准备 orchestrator 环境、创建目标完整 CPython patch 的独立子环境, 并由子解释器运行候选物化器。
3. `sd35_method_runtime_gpu`、`t2smark_sd35_gpu` 与三个 official-reference profile 使用相同隔离协议; 只有 `workflow_orchestrator` 候选允许在 Notebook 当前解释器中生成。

In [ ]:
import os
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
PROFILE_ID = "workflow_orchestrator"
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/SLM/dependency_lock_review_bundles")
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print({"profile_id": PROFILE_ID, "drive_output_dir": str(DRIVE_OUTPUT_DIR)})

In [ ]:
import re
import subprocess
import sys

repository_url = os.environ.get("SLM_WM_REPOSITORY_URL", "https://github.com/RICHAAARC/SLM-WM.git")
repository_commit = os.environ.get("SLM_WM_REPOSITORY_COMMIT", "")
if re.fullmatch(r"[0-9a-f]{40}", repository_commit) is None:
    raise ValueError("依赖锁审查前必须通过 SLM_WM_REPOSITORY_COMMIT 提供精确40位小写 Git SHA")
workspace_dir = Path(os.environ.get("SLM_WM_WORKSPACE_DIR", "/content/slm_wm_dependency_lock_review_repository"))
workspace_dir.parent.mkdir(parents=True, exist_ok=True)
if workspace_dir.exists() and not (workspace_dir / ".git").is_dir():
    raise RuntimeError("SLM_WM_WORKSPACE_DIR 已存在但不是 Git 仓库")
if not (workspace_dir / ".git").is_dir():
    subprocess.run(["git", "clone", repository_url, str(workspace_dir)], check=True)
status_before_checkout = subprocess.run(
    ["git", "status", "--porcelain=v1", "--untracked-files=all"],
    cwd=workspace_dir,
    check=True,
    capture_output=True,
    text=True,
).stdout
if status_before_checkout:
    raise RuntimeError("复用 Colab 工作区前必须先清理 Git 工作树")
subprocess.run(["git", "fetch", "origin", "--tags", "--prune"], cwd=workspace_dir, check=True)
subprocess.run(["git", "rev-parse", "--verify", repository_commit + "^{commit}"], cwd=workspace_dir, check=True)
subprocess.run(["git", "checkout", "--detach", repository_commit], cwd=workspace_dir, check=True)
os.chdir(workspace_dir)
if str(workspace_dir) not in sys.path:
    sys.path.insert(0, str(workspace_dir))
from paper_workflow.colab_utils.formal_execution import verify_and_publish_formal_execution

formal_execution_lock = verify_and_publish_formal_execution(workspace_dir, repository_commit)
print({"workspace_dir": str(workspace_dir), "formal_execution_lock": formal_execution_lock})

In [ ]:
review_command = [
    sys.executable,
    "scripts/write_dependency_lock_review_bundle.py",
    "--profile",
    PROFILE_ID,
    "--drive-output-dir",
    str(DRIVE_OUTPUT_DIR),
]
review_result = subprocess.run(review_command, check=True)
print({"profile_id": PROFILE_ID, "return_code": review_result.returncode})